Load paths of dataset, tokenizer, and model.

In [1]:
from datasets import load_dataset, load_from_disk
from transformers import AutoTokenizer, AutoModelForTokenClassification, Trainer, TrainingArguments

# Replace with the path or name of your saved model
model_path = "./models/dalembert-ner-finetuned_ep5"
dataset_path = "./data/ck_ner_dataset_hg"
tokenizer_path = "./models/dalembert-ner-finetuned_tokenizer"

### Load our medieval french dataset (locally) & model and tokenizer
dataset = load_from_disk(dataset_path)
tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
model = AutoModelForTokenClassification.from_pretrained(model_path)

##### Tokenization still needs to be done.

In [ ]:
from hg_functions import get_label_mappings, tokenize_and_align_labels_for_ner, compute_metrics

label_list, label_to_id, id_to_label = get_label_mappings(dataset)

eval_dataset = dataset["validation"].map(
    lambda x: tokenize_and_align_labels_for_ner(x, tokenizer, label_to_id),
    batched=False
)
training_args = TrainingArguments(
    output_dir = "./results", 
    per_device_eval_batch_size = 8,
    do_eval = True,
    logging_dir = "./logs",
    eval_accumulation_steps = 2
)

trainer = Trainer(
    model = model,
    args = training_args,
    tokenizer = tokenizer,
    compute_metrics = lambda p: compute_metrics(p, id_to_label)
)


Map:   0%|          | 0/12700 [00:00<?, ? examples/s]

Run evaluation

In [5]:
results = trainer.evaluate(eval_dataset)
print(results)

  0%|          | 0/1588 [00:00<?, ?it/s]

{'eval_loss': 0.018169540911912918, 'eval_precision': 0.9292389853137517, 'eval_recall': 0.9407716883755223, 'eval_f1': 0.9349697746840081, 'eval_accuracy': 0.9970777534928091, 'eval_runtime': 28.5147, 'eval_samples_per_second': 445.385, 'eval_steps_per_second': 55.691}


---

## 📚 Lesson Review — Evaluating the Fine-Tuned d'Alembert Model

### What we just did
We ran a full **quantitative evaluation** of the fine-tuned d'Alembert NER model on the validation split of our medieval French corpus, reporting precision, recall, F1, and accuracy.

---

### Reading the results

```
eval_precision : 0.9292  (92.9%)
eval_recall    : 0.9408  (94.1%)
eval_f1        : 0.9350  (93.5%)
eval_accuracy  : 0.9971  (99.7%)
```

These are **strong results** for a historical NER task. The gap between accuracy (99.7%) and F1 (93.5%) illustrates exactly why accuracy is not the right metric here — the corpus is dominated by `O` tokens, so predicting `O` for everything would already yield ~90% accuracy. F1 measures performance where it matters: on the actual entity spans.

---

### Key NLP concepts

#### 1. Precision, Recall, and F1 — the NER trinity

| Metric | Formula | Meaning |
|--------|---------|---------|
| **Precision** | TP / (TP + FP) | Of all entities the model predicted, how many are correct? |
| **Recall** | TP / (TP + FN) | Of all true entities in the corpus, how many did the model find? |
| **F1** | 2 × P × R / (P + R) | Harmonic mean — balances precision and recall |

Here recall (94.1%) slightly exceeds precision (92.9%), meaning the model is slightly more prone to **false positives** (hallucinating entities) than **false negatives** (missing entities). Whether this is acceptable depends on your use case: in archival indexing, missing a person name (`FN`) is worse than a spurious tag (`FP`).

#### 2. `seqeval` entity-level scoring
The evaluation uses `seqeval`, which treats a prediction as correct only if **both the span boundaries and the label match exactly**. This is stricter than token-level accuracy:

```
Gold:  [B-PERSON, I-PERSON, I-PERSON, O]   → "Jehan de Flores"
Pred:  [B-PERSON, I-PERSON, O, O]           → "Jehan de"
→ seqeval counts this as 0 TP, 1 FN, 1 FP
```

Partial overlaps score zero — the full span must be right.

#### 3. Re-tokenization at evaluation time
Even though the model was trained with tokenization baked in, evaluation requires re-running `tokenize_and_align_labels_for_ner()` on the validation set. This is because:
- `Trainer.evaluate()` expects tokenized input tensors, not raw text
- The label alignment (`-100` masking for subwords) must be consistent between training and evaluation

Using a shared helper function (`hg_functions.py`) ensures this consistency.

#### 4. `eval_accumulation_steps=2` — why this matters
Evaluation on the full validation set accumulates logits and labels in memory before computing metrics. `eval_accumulation_steps=2` processes the evaluation in chunks of 2 batches at a time, preventing GPU/CPU memory overflow when the validation set is large (12,700 examples here).

#### 5. The full pipeline in context
This notebook is the final step in a complete NLP pipeline:

```
Raw CoNLL corpus
    ↓  01_data_import      — parse & structure
    ↓  02_prepare_spacy    — character spans + spaCy format
    ↓  03_finetune_dalembert — fine-tune transformer
    ↓  04_test_spacy       — qualitative spaCy testing
    ↓  05_test_dalembert   ← YOU ARE HERE — quantitative evaluation
```

An F1 of **93.5%** on medieval French NER — a low-resource, high-variance domain — is a strong result, reflecting both the quality of the annotation in the PRESTO corpus and the value of using a domain-adapted pre-trained model (d'Alembert).

---

### Things to think about
- The model achieves 93.5% overall F1. But this is an *average* across all entity types. Which entity types (PERSON, LOCATION, AMOUNT, etc.) do you expect to score highest and lowest, and why?
- `eval_loss: 0.018` is very low. Could this indicate overfitting? What additional evidence would you look for?
- If you were to deploy this model on a new corpus of medieval French texts with different authors and time periods, what degradation in performance would you expect, and how would you measure it?
